# Model selection with cross-validation

We compare three candidate models on the training set with **10-fold cross-validation**: every model is trained and evaluated ten times, each time on a different fold, and we look at the mean and spread of the ten RMSEs (see [Cross-validation](../notes/cross_validation.ipynb)). The test set is not used for this comparison; using it here would be [model selection leakage](../notes/data_leakage.ipynb#Model-selection-leakage).

In [ ]:
from utils.load_california import load_housing_data
from utils.housing_preprocessing import get_preprocessing_pipeline

X_train, X_test, y_train, y_test = load_housing_data()
preprocessing = get_preprocessing_pipeline(n_clusters=10)  # provisional value, tuned in the next notebook

## Cross-validation setup

The same `KFold` object is used for all models, so each one is evaluated on exactly the same ten splits. `shuffle=True` prevents any ordering of the rows from biasing the folds, and `random_state` makes the folds reproducible.

In [ ]:
import pandas as pd
from sklearn.model_selection import KFold, cross_val_score

cv = KFold(n_splits=10, shuffle=True, random_state=42)

def cv_rmse(model):
    """RMSE of the model on each of the validation folds."""
    # scoring expects a utility (higher is better), so scikit-learn returns the negative RMSE
    return -cross_val_score(model, X_train, y_train, scoring="neg_root_mean_squared_error", cv=cv)

## Linear regression

Each candidate is a full pipeline: the preprocessing followed by the model. Inside `cross_val_score` the preprocessing is refitted on the training folds of every split, which is what keeps the validation folds clean.

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression

lin_reg = make_pipeline(preprocessing, LinearRegression())
lin_rmses = cv_rmse(lin_reg)
pd.Series(lin_rmses).describe()

count       10.000000
mean     70208.295816
std       3565.221286
min      66042.802394
25%      68404.352133
50%      69653.157656
75%      71136.752951
max      78799.382425
dtype: float64

## Decision tree

In [ ]:
from sklearn.tree import DecisionTreeRegressor

tree_reg = make_pipeline(preprocessing, DecisionTreeRegressor(random_state=42))
tree_rmses = cv_rmse(tree_reg)
pd.Series(tree_rmses).describe()

count       10.000000
mean     66987.418379
std       2282.723283
min      64515.964221
25%      65039.293098
50%      66528.619899
75%      68269.938325
max      71379.365812
dtype: float64

## Random forest

*Random Forest* trains many decision trees (***ensemble learning***) on random subsets of the data and features and averages their predictions.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

forest_reg = make_pipeline(preprocessing, RandomForestRegressor(random_state=42))
forest_rmses = cv_rmse(forest_reg)
pd.Series(forest_rmses).describe()

count       10.000000
mean     47339.800646
std       1817.506377
min      44796.373845
25%      46272.551225
50%      47186.956219
75%      48615.602528
max      50638.943557
dtype: float64

## Comparison

In [ ]:
pd.DataFrame({
    "Linear regression": lin_rmses,
    "Decision tree": tree_rmses,
    "Random forest": forest_rmses,
}).agg(["mean", "std"]).T.round(0)

,mean,std
Linear regression,70208.0,3565.0
Decision tree,66987.0,2283.0
Random forest,47340.0,1818.0


The random forest is clearly the best of the three on validation, with a mean RMSE around 47,000 USD. That is still a large error for house values whose median is about 180,000 USD, but it is the model we will keep. The next step is to tune its hyperparameters, together with the number of geographic clusters of the preprocessing, in [`07_hyperparameter_tuning.ipynb`](07_hyperparameter_tuning.ipynb).